# 01 — Data Processing

Loads PADS raw data, filters to RightWrist + Relaxed task (HC vs PD),
applies bandpass filter, segments into windows, and saves four files:

```
processed/windows.npy       — (N, 6, 200)
processed/labels.npy        — (N,)  0=HC  1=PD
processed/subject_ids.npy   — (N,)  integer subject ID
processed/fold_splits.pkl   — 5-fold CV subject-level splits
```

Run once. All downstream notebooks load from these files.

In [ ]:
import sys, os, os.path as osp, json
import numpy as np
import pandas as pd

REPO_DIR = '/kaggle/working/federated-subset-scanning-pads'
if not osp.exists(REPO_DIR):
    os.system('git clone https://github.com/isaacmutuma/federated-subset-scanning-pads.git ' + REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
from src.data.preprocessing import bandpass_filter, segment_windows
from src.data.folds import generate_fold_splits
print('Imports OK')

In [ ]:
def find_pads_root(base):
    for root, dirs, files in os.walk(base):
        if 'movement' in dirs and 'patients' in dirs:
            return root
    return None

BASE_PATH = find_pads_root('/kaggle/input')
assert BASE_PATH is not None, 'PADS dataset not found — attach it to this notebook'

OBSERVATION_DIR = osp.join(BASE_PATH, 'movement')
PATIENTS_DIR    = osp.join(BASE_PATH, 'patients')
OUTPUT_DIR      = '/kaggle/working/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'PADS root:  {BASE_PATH}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Contents:   {os.listdir(BASE_PATH)}')

In [ ]:
def get_patient_label(patient_id, patients_dir):
    path = osp.join(patients_dir, f'patient_{int(patient_id):03d}.json')
    with open(path) as f:
        return json.load(f).get('condition')

def build_manifest(patients_dir, observation_dir, n_patients=469):
    rows = []
    for pat_num in range(1, n_patients + 1):
        patient_id = f'{pat_num:03d}'
        label      = get_patient_label(pat_num, patients_dir)
        obs_path   = osp.join(observation_dir, f'observation_{patient_id}.json')
        with open(obs_path) as f:
            obs = json.load(f)
        for session in obs['session']:
            for record in session['records']:
                rows.append({
                    'patient_id': patient_id,
                    'label':      label,
                    'task':       session.get('record_name'),
                    'wrist':      record.get('device_location'),
                    'filepath':   record.get('file_name'),
                })
    return pd.DataFrame(rows)

manifest = build_manifest(PATIENTS_DIR, OBSERVATION_DIR)
print(f'Total rows: {len(manifest)}')
print(manifest['label'].value_counts())

In [ ]:
all_labels_found = manifest['label'].unique().tolist()
HC_STR = next(l for l in all_labels_found if 'healthy' in l.lower() or l == 'HC')
PD_STR = next(l for l in all_labels_found if 'parkinson' in l.lower() or l == 'PD')
LABEL_MAP = {HC_STR: 0, PD_STR: 1}
print(f'HC="{HC_STR}"  PD="{PD_STR}"')

filtered = manifest[
    (manifest['task']  == 'Relaxed') &
    (manifest['wrist'] == 'RightWrist') &
    (manifest['label'].isin([HC_STR, PD_STR]))
].copy()

print(f'Filtered rows: {len(filtered)}')
print(filtered['label'].value_counts())
print(f'Unique subjects: {filtered["patient_id"].nunique()}')

In [ ]:
def load_timeseries(filepath):
    df = pd.read_csv(filepath, header=None)
    return df.iloc[:, 1:7].values.T.astype(np.float32)

FS, WINDOW_SIZE, LOWCUT, HIGHCUT, N_WINDOWS = 100.0, 200, 1.0, 20.0, 10
all_windows, all_labels_out, all_subject_ids = [], [], []
skipped = 0

for _, row in filtered.iterrows():
    filepath = osp.join(BASE_PATH, 'movement', row['filepath'])
    if not osp.exists(filepath): skipped += 1; continue
    try: signal = load_timeseries(filepath)
    except Exception: skipped += 1; continue
    if signal.shape[1] < WINDOW_SIZE * N_WINDOWS: skipped += 1; continue

    sig_filt = bandpass_filter(signal, lowcut=LOWCUT, highcut=HIGHCUT, fs=FS)
    windows  = segment_windows(sig_filt, window_size=WINDOW_SIZE, step=WINDOW_SIZE)[:N_WINDOWS]
    if len(windows) < N_WINDOWS: skipped += 1; continue

    all_windows.append(windows)
    all_labels_out.extend([LABEL_MAP[row['label']]] * N_WINDOWS)
    all_subject_ids.extend([int(row['patient_id'])] * N_WINDOWS)

print(f'Skipped: {skipped}  |  Processed: {len(all_windows)} subjects')

In [ ]:
windows     = np.concatenate(all_windows, axis=0)
labels      = np.array(all_labels_out,  dtype=np.int64)
subject_ids = np.array(all_subject_ids, dtype=np.int64)

print(f'Windows: {windows.shape}  HC={(labels==0).sum()}  PD={(labels==1).sum()}')
assert windows.shape[1:] == (6, 200)
assert len(windows) == len(labels) == len(subject_ids)
print('Checks passed.')

In [ ]:
unique_subjects = np.unique(subject_ids)
subject_labels  = np.array([labels[subject_ids == s][0] for s in unique_subjects])

folds = generate_fold_splits(
    subject_ids=unique_subjects,
    subject_labels=subject_labels,
    n_splits=5, random_state=42, val_fraction=0.2,
    save_path=osp.join(OUTPUT_DIR, 'fold_splits.pkl'),
)

for i, fold in enumerate(folds):
    print(f'Fold {i+1}: train={len(fold["train_subjects"])} val={len(fold["val_subjects"])} test={len(fold["test_subjects"])}')

In [ ]:
np.save(osp.join(OUTPUT_DIR, 'windows.npy'),     windows)
np.save(osp.join(OUTPUT_DIR, 'labels.npy'),      labels)
np.save(osp.join(OUTPUT_DIR, 'subject_ids.npy'), subject_ids)

for fname in ['windows.npy', 'labels.npy', 'subject_ids.npy', 'fold_splits.pkl']:
    path = osp.join(OUTPUT_DIR, fname)
    print(f'{fname}  ({osp.getsize(path)/1e6:.1f} MB)')

print('Done. Ready for notebook 03.')